# Data Preparation - Olist Marketplace
Daten strukturiert und reproduzierbar verarbeiten

## Verbindung mit Duckdb

In [1]:
import duckdb
from pathlib import Path
import pandas as pd

# Projektpfade
RAW_PATH = Path("../data/raw/brazilian-ecommerce")
DB_PATH = Path("../data/processed/olist.duckdb")

# Verbindung zur DuckDB als Datei (persistente DB statt in-memory)
con = duckdb.connect(DB_PATH.as_posix())

# Hilfsfunktion für SQL-Abfragen
def sql(query: str):
    return con.execute(query).df()

# Alle CSV-Dateien in DuckDB als Tabellen speichern (persistent in olist.duckdb)
for file in RAW_PATH.glob("*.csv"):
    table_name = file.stem.replace("olist_", "").replace("_dataset", "")

    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM read_csv_auto('{file.as_posix()}');
    """)

# Alle Tabellen anzeigen
sql("SHOW TABLES")

,name
0,customers
1,geolocation
2,order_items
3,order_payments
4,order_reviews
5,orders
6,product_category_name_translation
7,products
8,sellers


## Erstellung der Dataframes pro Kernaufgabe

### EDA für erste Kernaufgabe

In [2]:
# Dataframe für Aufgabe 1. für EDA
df_rfm_eda = sql("""
SELECT 
    c.customer_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    c.customer_zip_code_prefix,
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    oi.price,
    oi.freight_value,
    Count(oi.product_id) AS product_count,
    
FROM customers AS c
JOIN orders o 
        ON c.customer_id = o.customer_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
GROUP BY c.customer_id, o.order_id, o.order_purchase_timestamp, 
                 o.order_approved_at, oi.price, oi.freight_value, 
                 c.customer_unique_id, c.customer_city, c.customer_state, 
                 c.customer_zip_code_prefix, o.order_status
    """)

In [3]:
df_rfm_eda

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
0,81e08b08e5ed4472008030d70327c71f,0e764fc1a13e47e900c3d59a989753e8,juiz de fora,MG,36045,688052146432ef8253587b930b01a06d,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,199.00,3.12,1
1,93ada7a24817edda9f4ab998fa823d16,cd148470c375939669971e8a032b16b4,ribeirao preto,SP,14091,cadbb3657dac2dbbd5b84b12e7b78aad,delivered,2018-02-27 12:55:42,2018-03-01 02:48:54,394.90,14.89,1
2,167b9485947ed0a354a3f8dad04eb199,548a09978548d2e347d494793e34c797,barueri,SP,06462,f4471dae8c482f51aa1826cd9f5d4433,delivered,2018-07-05 18:40:47,2018-07-05 18:55:15,110.32,12.53,1
3,f7398fc942c8fa80e5419ae52e49f7fb,d01cf8c6c7c836c5dd9320585928f42b,barueri,SP,06414,ce9feeba53c652dd6569cca62e2bb287,delivered,2018-04-15 19:42:06,2018-04-15 19:55:20,45.00,11.86,1
4,7df71d1652cfba8b2a70cf665c960f74,cb644c8c88c4705149fc0c7dd12d596d,palotina,PR,85950,a03ad7057fae696dd18f8967826c209f,delivered,2017-06-05 13:24:46,2017-06-07 19:42:11,52.88,25.21,1
...,...,...,...,...,...,...,...,...,...,...,...,...
101565,a2903717cca50b4ac3b41a2074f1020c,b318a9471f7e803303ab808bfd4605f3,itaborai,RJ,24855,91c67084c8053e40be5dc547d3e9a01e,delivered,2017-07-03 18:39:32,2017-07-03 18:50:11,130.99,28.65,1
101566,d7cf6e6f535ab1be102e3e7ab5185139,659d92dc24d2773bb6c74985a14d2221,taboao da serra,SP,06768,b80264e091caa14505dc8e800902c80c,delivered,2018-02-22 23:13:55,2018-02-22 23:28:21,103.85,34.22,1
101567,6e875338f7a2933dcabb08b9404c3f72,83be9597caaf2ff1ad90bace79385397,sao paulo,SP,04078,4d288f2f1f3b59a97c364a2bf728b912,delivered,2018-06-14 14:54:38,2018-06-14 15:20:49,55.00,11.72,1
101568,4e0d465bf7a56e22b91d75f01e5c8b0e,792f1a827e7ca88e7f76678562e5fc7b,juquia,SP,11800,c48b937c24e7b161a3685f1a8bc963b7,delivered,2018-01-15 22:34:23,2018-01-15 22:51:33,49.90,13.54,1


In [4]:
df_rfm_eda.describe()

,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
count,101570,101556,101570.000000,101570.000000,101570.000000
mean,2018-01-01 00:42:04.180673,2018-01-01 12:06:22.988666,124.922151,20.140526,1.109087
min,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000,1.000000
25%,2017-09-13 08:05:38,2017-09-13 17:27:48.500000,40.800000,13.160000,1.000000
50%,2018-01-19 16:57:45,2018-01-20 09:09:20.500000,79.000000,16.340000,1.000000
75%,2018-05-04 23:26:44,2018-05-05 12:55:31.500000,139.530000,21.260000,1.000000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.680000,20.000000
std,NaN,NaN,189.479405,15.901388,0.474137


#### Auffälligkeiten

1. Bei order_purchase_timestamp und order_approved_at scheint es NaN werte zu geben
2. 

In [5]:
df_rfm_eda.dtypes

customer_id                         object
customer_unique_id                  object
customer_city                       object
customer_state                      object
customer_zip_code_prefix            object
order_id                            object
order_status                        object
order_purchase_timestamp    datetime64[us]
order_approved_at           datetime64[us]
price                              float64
freight_value                      float64
product_count                        int64
dtype: object

In [6]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = {'customer_id': 'category', 
              'customer_unique_id': 'category', 
              'customer_city': 'category', 
              'customer_state': 'category', 
              'customer_zip_code_prefix': 'category', 
              'order_id': 'category', 
              'order_status': 'category', 
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_approved_at': 'datetime64[s]', 
              'price': 'float32',
              'freight_value': 'float32',
              'product_count': 'int16', }
df_rfm_eda = df_rfm_eda.astype(col_dtypes)
df_rfm_eda.dtypes

customer_id                      category
customer_unique_id               category
customer_city                    category
customer_state                   category
customer_zip_code_prefix         category
order_id                         category
order_status                     category
order_purchase_timestamp    datetime64[s]
order_approved_at           datetime64[s]
price                             float32
freight_value                     float32
product_count                       int16
dtype: object

In [7]:
df_rfm_eda.describe()

,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
count,101570,101556,101570.000000,101570.000000,101570.000000
mean,2018-01-01 00:42:04,2018-01-01 12:06:22,124.922150,20.140526,1.109087
min,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000,1.000000
25%,2017-09-13 08:05:38,2017-09-13 17:27:48,40.799999,13.160000,1.000000
50%,2018-01-19 16:57:45,2018-01-20 09:09:20,79.000000,16.340000,1.000000
75%,2018-05-04 23:26:44,2018-05-05 12:55:31,139.529995,21.260000,1.000000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.679993,20.000000
std,NaN,NaN,189.479401,15.901388,0.474137


In [8]:
df_rfm_eda.isna().sum()

customer_id                  0
customer_unique_id           0
customer_city                0
customer_state               0
customer_zip_code_prefix     0
order_id                     0
order_status                 0
order_purchase_timestamp     0
order_approved_at           14
price                        0
freight_value                0
product_count                0
dtype: int64

In [9]:
df_rfm_eda[df_rfm_eda.isna().any(axis=1)].head(15)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
10398,07a2a7e0f63fd8cb757ed77d4245623c,79af1bbf230a2630487975aa5d7d6220,paraisopolis,MG,37660,51eb2eebd5d76a24625b31c33dd41449,delivered,2017-02-18 15:52:27,NaT,59.900002,17.160000,1
22917,a3d3c38e58b9d2dfb9207cab690b6310,5a4fa4919cbf2b049e72be460a380e5b,abaete,MG,35620,2eecb0d85f281280f79fa00f9cec1a95,delivered,2017-02-17 17:21:55,NaT,135.000000,19.230000,1
31813,4c1ccc74e00993733742a3c786dc3c1f,91efb7fcabc17925099dced52435837f,novo hamburgo,RS,93548,8a9adc69528e1001fc68dd0aaebbb54a,delivered,2017-02-18 12:45:31,NaT,379.000000,17.860001,1
39377,74bebaf46603f9340e3b50c6b086f992,f79be7c08dd24b72d34634f1b89333a4,sao jose de ribamar,MA,65110,2babbb4b15e6d2dfe95e2de765c97bce,delivered,2017-02-18 17:15:03,NaT,79.989998,26.820000,1
40274,0bf35cac6cc7327065da879e2d90fae8,c4c0011e639bdbcf26059ddc38bd3c18,varzea paulista,SP,13225,d77031d6a3c8a52f019764e68f211c69,delivered,2017-02-18 11:04:19,NaT,28.990000,10.960000,1
40805,d85919cb3c0529589c6fa617f5f43281,c094ac95fcd52f821809ec232a7a6956,sao vendelino,RS,95795,3c0b8706b065f9919d0505d3b3343881,delivered,2017-02-17 15:53:27,NaT,133.990005,23.200001,1
49243,2127dc6603ac33544953ef05ec155771,8a9a08c7ca8900a200d83cf838a07e0b,cotia,SP,06708,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,NaT,309.899994,39.110001,1
49572,684cb238dc5b5d6366244e0e0776b450,6ff8b0d7b35d5c945633b8d60165691b,santos,SP,11030,c1d4211b3dae76144deccd6c74144a88,delivered,2017-01-19 12:48:08,NaT,39.990002,14.520000,1
50204,f67cd1a215aae2a1074638bbd35a223a,bc1896dc77f49e6dec880445a9b443a3,rio de janeiro,RJ,21020,88083e8f64d95b932164187484d90212,delivered,2017-02-18 22:49:19,NaT,49.000000,14.520000,2
57646,68d081753ad4fe22fc4d410a9eb1ca01,2e0a2166aa23da2472c6a60c4af6f7a6,sao paulo,SP,03573,d69e5d356402adc8cf17e08b5033acfb,delivered,2017-02-19 01:28:47,NaT,149.800003,13.630000,1


In [10]:
pd.crosstab(df_rfm_eda['order_status'], df_rfm_eda['order_approved_at'].isna())

order_approved_at,False,True
order_status,,
approved,2,0
canceled,464,0
delivered,99341,14
invoiced,319,0
processing,304,0
shipped,1119,0
unavailable,7,0


In [11]:
df_rfm_eda['order_approved_at'] = df_rfm_eda['order_approved_at'].fillna(
    pd.to_datetime(df_rfm_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_rfm_eda[df_rfm_eda.isna().any(axis=1)].head(15)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,product_count


In [12]:
print("Gesamte Duplikate:", df_rfm_eda.duplicated().sum())

Gesamte Duplikate: 0


In [13]:
test = pd.crosstab(df_rfm_eda['order_status'], df_rfm_eda['order_id']).T
test


order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable
order_id,,,,,,,
00010242fe8c5a6d1ba2dd792cb16214,0,0,1,0,0,0,0
00018f77f2f0320c557190d7a144bdd3,0,0,1,0,0,0,0
000229ec398224ef6ca0657da4fc703e,0,0,1,0,0,0,0
00024acbcdf0a6daa1e931b038114c75,0,0,1,0,0,0,0
00042b26cf59d7ce69dfabb4e55b4fd9,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...
fffc94f6ce00a00581880bf54a75a037,0,0,1,0,0,0,0
fffcd46ef2263f404302a634eb57f7eb,0,0,1,0,0,0,0
fffce4705a9662cd70adb13d4a31832d,0,0,1,0,0,0,0


In [14]:
test['sum_status']=test.sum(axis=1)

In [15]:
test.loc[test['sum_status']!=1, :] 

order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable,sum_status
order_id,,,,,,,,
002f98c0f7efd42638ed6100ca699b42,0,0,2,0,0,0,0,2
005d9a5423d47281ac463a968b3936fb,0,0,2,0,0,0,0,2
00946f674d880be1f188abc10ad7cf46,0,0,2,0,0,0,0,2
0097f0545a302aafa32782f1734ff71c,0,0,2,0,0,0,0,2
00bcee890eba57a9767c7b5ca12d3a1b,0,0,2,0,0,0,0,2
...,...,...,...,...,...,...,...,...
ffb18bf111fa70edf316eb0390427986,0,0,2,0,0,0,0,2
ffb8f7de8940249a3221252818937ecb,0,0,3,0,0,0,0,3
ffb9a9cd00c74c11c24aa30b3d78e03b,0,0,3,0,0,0,0,3


####  Auffälligkeiten
1. Die Nan Werte machen einen sehr geringen Anteil aus. 
2. Da die Zeilen in denen sich die NaN werte befinden, den Order Status delivered haben, werde ich die Daten berücksichten, da der Kauf stattgefunden.



Für die Auswertung von Aufgabe 1. ist der order_status sehr wichtig, da ich nur reale Bestellungen betrachten will.

Deswegen schau ich mir erstmal an wie sich die NaNs zu den relevanten Order Status verhalten

Für die RFM Analyse brauche ich nur die approved, delivered, invoiced, processing und shipped order_status
Daher kann ich canceled, created und unavailable erstmal rausnehmen, da dieser order_status für die Aufgabe nicht relevant ist.

In [16]:
valid_rfm_status = ['delivered', 'shipped', 'processing', 'invoiced', 'approved']
df_rfm_eda = df_rfm_eda[df_rfm_eda['order_status'].isin(valid_rfm_status)]

In [17]:
test.loc[test['sum_status']!=1, :].sort_values('sum_status', ascending=False)

order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable,sum_status
order_id,,,,,,,,
ca3625898fbd48669d50701aba51cd5f,0,0,7,0,0,0,0,7
cf5c8d9f52807cb2d2f0a0ff54c478da,0,0,6,0,0,0,0,6
5a3b1c29a49756e75f1ef513383c0c12,0,0,6,0,0,0,0,6
b436eb981676e54c0bc9bcade0e079c4,0,0,5,0,0,0,0,5
bb82809ea3ca9f3edbe589b60e14e0cb,0,0,5,0,0,0,0,5
...,...,...,...,...,...,...,...,...
59b67c775c6a905fc4faac69ca74b5cb,0,0,2,0,0,0,0,2
59bccab4e9193a9229f7d1b73fcb47c3,0,0,2,0,0,0,0,2
59c0ed646a3b30d4054298988188486f,0,0,2,0,0,0,0,2


## Filterung nach der Bestellung mit den meisten Duplikaten

In [18]:
order_id = 'ca3625898fbd48669d50701aba51cd5f'

# 1. Filter auf diese Order_ID
order_data = df_rfm_eda[df_rfm_eda['order_id'] == order_id]

order_data.head(63)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
10713,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,309.000000,1.84,1
11896,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,56.000000,3.68,2
15392,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,159.000000,3.67,2
63079,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,33.900002,1.84,1
77493,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,109.900002,0.15,1
85560,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,63.700001,0.15,1
87109,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,95.900002,0.15,2


In [19]:
df_rfm_eda.describe()

,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
count,101099,101099,101099.000000,101099.000000,101099.000000
mean,2018-01-01 04:49:34,2018-01-01 15:09:29,124.649025,20.140503,1.108824
min,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000,1.000000
25%,2017-09-13 11:58:02,2017-09-13 20:10:21,40.799999,13.180000,1.000000
50%,2018-01-19 16:33:57,2018-01-20 09:08:37,79.000000,16.350000,1.000000
75%,2018-05-05 07:49:38,2018-05-05 14:13:51,139.000000,21.260000,1.000000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.679993,20.000000
std,NaN,NaN,188.526642,15.891350,0.473023


In [20]:
con.execute("CREATE TABLE customer_rfm AS SELECT * FROM df_rfm_eda")

### EDA für zweite Kernaufgabe

In [21]:
# Dataframe für Aufgabe 2. für EDA
df_pc_eda = sql("""
SELECT 
    p.product_id,
    pcnt.product_category_name_english,
    Count(o.order_id) AS order_count,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    oi.price,
    oi.freight_value,
    r.review_score
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id 
JOIN orders o ON oi.order_id = o.order_id
JOIN product_category_name_translation pcnt ON pcnt.product_category_name = p.product_category_name
LEFT JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status IN ('delivered', 'shipped', 'processing', 'invoiced', 'approved')
GROUP BY p.product_id, pcnt.product_category_name_english, o.order_status, o.order_purchase_timestamp,
         o.order_approved_at, oi.price, oi.freight_value, r.review_score
    """)

In [22]:
df_pc_eda

,product_id,product_category_name_english,order_count,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,review_score
0,fd25ab760bfbba13c198fa3b4f1a0cd3,sports_leisure,2,delivered,2018-01-11 15:30:49,2018-01-11 15:47:59,185.00,13.63,4
1,154e7e31ebfa092203795c972e5804a6,health_beauty,1,delivered,2017-10-02 11:17:07,2017-10-02 11:28:29,23.99,14.10,4
2,d0fe4295267f15ccaceac4fb233d8c9a,computers_accessories,1,delivered,2018-04-07 15:24:30,2018-04-07 15:35:10,45.90,8.82,5
3,5cbd407f3315b628a89206fbc140f6c8,market_place,1,delivered,2018-03-28 16:16:20,2018-03-30 03:08:56,19.90,7.39,5
4,a0a6b0afd47416d62bb25892c68b6296,garden_tools,1,delivered,2017-12-29 12:29:00,2017-12-29 12:46:44,62.40,15.88,5
...,...,...,...,...,...,...,...,...,...
100703,53759a2ecddad2bb87a079a1f1519f73,garden_tools,1,delivered,2017-12-29 11:10:34,2017-12-29 11:27:32,49.90,17.66,<NA>
100704,99a4788cb24856965c36a24e339b6058,bed_bath_table,1,delivered,2017-08-19 20:25:59,2017-08-22 04:05:17,89.90,2.41,<NA>
100705,a96e543f9cfb2ecdb07c59321ff0018c,toys,1,delivered,2017-08-09 16:43:18,2017-08-09 17:03:52,119.90,16.47,<NA>
100706,440cb991a95f6a9c8918c112433f4d69,bed_bath_table,1,delivered,2017-12-28 15:32:45,2017-12-28 15:47:35,207.00,18.70,<NA>


In [23]:
df_pc_eda.describe()

,order_count,order_purchase_timestamp,order_approved_at,price,freight_value,review_score
count,100708.000000,100708,100695,100708.000000,100708.000000,99940.0
mean,1.103597,2018-01-01 16:12:59.141210,2018-01-02 03:32:58.710333,124.151016,20.142170,4.088883
min,1.000000,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000,1.0
25%,1.000000,2017-09-13 17:09:08,2017-09-14 02:45:40,40.140000,13.180000,4.0
50%,1.000000,2018-01-20 13:59:55.500000,2018-01-20 20:00:10,78.000000,16.360000,5.0
75%,1.000000,2018-05-05 21:22:12.750000,2018-05-06 13:50:11,139.000000,21.260000,5.0
max,20.000000,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.680000,5.0
std,0.461837,NaN,NaN,187.484910,15.898289,1.342309


In [24]:
df_pc_eda.dtypes

product_id                               object
product_category_name_english            object
order_count                               int64
order_status                             object
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
price                                   float64
freight_value                           float64
review_score                              Int64
dtype: object

In [25]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = { 
              'product_id': 'category',
              'product_category_name_english': 'category', 
              'order_count': 'Int16', 
              'order_status': 'category', 
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_approved_at': 'datetime64[s]', 
              'price': 'float32',
              'freight_value': 'float32',
              'review_score': 'category',}
df_pc_eda = df_pc_eda.astype(col_dtypes)
df_pc_eda.dtypes

product_id                            category
product_category_name_english         category
order_count                              Int16
order_status                          category
order_purchase_timestamp         datetime64[s]
order_approved_at                datetime64[s]
price                                  float32
freight_value                          float32
review_score                          category
dtype: object

In [26]:
df_pc_eda.describe()

,order_count,order_purchase_timestamp,order_approved_at,price,freight_value
count,100708.0,100708,100695,100708.000000,100708.000000
mean,1.103597,2018-01-01 16:12:59,2018-01-02 03:32:58,124.151009,20.142170
min,1.0,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000
25%,1.0,2017-09-13 17:09:08,2017-09-14 02:45:40,40.139999,13.180000
50%,1.0,2018-01-20 13:59:55,2018-01-20 20:00:10,78.000000,16.360001
75%,1.0,2018-05-05 21:22:12,2018-05-06 13:50:11,139.000000,21.260000
max,20.0,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.679993
std,0.461837,NaN,NaN,187.484909,15.898289


In [27]:
df_pc_eda.isna().sum()

product_id                         0
product_category_name_english      0
order_count                        0
order_status                       0
order_purchase_timestamp           0
order_approved_at                 13
price                              0
freight_value                      0
review_score                     768
dtype: int64

In [28]:
df_pc_eda['order_approved_at'] = df_pc_eda['order_approved_at'].fillna(
    pd.to_datetime(df_pc_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_pc_eda[df_pc_eda.isna().any(axis=1)].head(15)

,product_id,product_category_name_english,order_count,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,review_score
1552,6f3b5b605d91b7439c5e3f5a8dffeea7,watches_gifts,1,delivered,2018-03-19 14:28:12,2018-03-19 14:49:35,165.000000,19.030001,NaN
1553,154b6772d166a9c3398661ea26350dc6,sports_leisure,1,delivered,2018-03-07 12:27:19,2018-03-09 18:29:23,399.899994,37.930000,NaN
1554,ebc714d9f070a89f4d411982ea9670fb,toys,1,delivered,2017-12-06 10:51:17,2017-12-06 11:14:49,499.989990,23.990000,NaN
1555,165f86fe8b799a708a20ee4ba125c289,cool_stuff,1,delivered,2018-04-12 15:15:43,2018-04-12 15:30:49,169.990005,15.180000,NaN
1556,fb55982be901439613a95940feefd9ee,stationery,1,delivered,2017-12-20 21:44:31,2017-12-20 21:56:28,79.000000,13.570000,NaN
1557,1fa0faff5eafb13003f9559ebe6becb3,watches_gifts,1,delivered,2018-03-16 19:39:14,2018-03-16 19:55:22,59.000000,15.290000,NaN
1558,d6fe3b4ddecd4a8393c6a1385de3bfb6,office_furniture,3,delivered,2017-03-09 23:42:58,2017-03-09 23:42:58,199.990005,34.439999,NaN
1559,a92930c327948861c015c919a0bcb4a8,watches_gifts,1,delivered,2017-06-19 01:46:34,2017-06-20 11:35:25,78.000000,7.800000,NaN
1560,3d77287739b6bf1ac163ce2d77570ada,furniture_decor,2,delivered,2017-04-21 00:11:35,2017-04-21 01:05:19,449.899994,124.540001,NaN
1561,e0f55a53bebaae74a51a8fb9639681d6,baby,2,delivered,2018-05-04 09:49:02,2018-05-04 10:11:27,18.400000,12.790000,NaN


In [29]:
print("Gesamte Duplikate:", df_pc_eda.duplicated().sum())

Gesamte Duplikate: 0


In [30]:
con.execute("CREATE TABLE product_category AS SELECT * FROM df_pc_eda")

### EDA für dritte Kernaufgabe

In [31]:
# Dataframe für Aufgabe 2. für EDA
df_service_eda = sql("""
SELECT 
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
        r.review_id,
        r.review_score,
        r.review_comment_title,
        r.review_comment_message,
        r.review_creation_date,
        r.review_answer_timestamp,
    oi.product_id,
    pcnt.product_category_name_english,
    Count(oi.product_id) AS product_count,                
    s.seller_id,
    s.seller_city,
    s.seller_state,
    c.customer_city,
    c.customer_state,              
    pcnt.product_category_name_english,
     
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
JOIN sellers s
        ON oi.seller_id = s.seller_id
JOIN customers c 
        ON c.customer_id = o.customer_id
JOIN products p 
        ON oi.product_id = p.product_id
JOIN product_category_name_translation pcnt 
        ON pcnt.product_category_name = p.product_category_name
WHERE o.order_status IN ('delivered')
Group BY o.order_id, o.order_status, o.order_purchase_timestamp, o.order_approved_at,
         o.order_delivered_carrier_date, o.order_delivered_customer_date, o.order_estimated_delivery_date,
         r.review_id, r.review_score, r.review_comment_title, r.review_comment_message, 
                     r.review_creation_date, r.review_answer_timestamp,
         oi.product_id, s.seller_id, s.seller_city, s.seller_state, 
                     c.customer_city, c.customer_state, pcnt.product_category_name_english
    """)

In [32]:
df_service_eda

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,...,review_answer_timestamp,product_id,product_category_name_english,product_count,seller_id,seller_city,seller_state,customer_city,customer_state,product_category_name_english_1
0,1be187af7e77d6ee0671be02c229b679,delivered,2018-05-30 10:12:07,2018-05-30 10:30:18,2018-06-01 12:54:00,2018-06-14 20:46:56,2018-07-12,696a80628f16220233a31cab1ce3ef30,2,None,...,2018-06-16 00:35:17,966746deb2173a72d95ca8b80223f014,bed_bath_table,1,1900267e848ceeba8fa32d80c1a5f5a8,ibitinga,SP,uberlandia,MG,bed_bath_table
1,e11dead0c183b7b8da09f6b3039ea730,delivered,2017-02-09 10:40:05,2017-02-09 10:55:14,2017-02-10 11:09:01,2017-02-14 13:08:44,2017-03-10,3dacc0ba7e4bcfe150e398b01637472b,3,None,...,2017-02-16 17:19:49,2fcbaee260e3146369f5e4f2b2abe551,garden_tools,1,5fd924b4836098a5be0ecf81ba054ce0,sao paulo,SP,belo horizonte,MG,garden_tools
2,905f7a43f18905597465564ae6deb958,delivered,2018-08-10 22:59:49,2018-08-14 04:45:20,2018-08-15 05:41:00,2018-08-28 21:41:40,2018-08-30,13164d9da17abb1a513eb773bff4f274,1,Recebi pedido incompleto,...,2018-10-02 20:44:56,51ce083cd2b9078656a94655ab45b8a4,bed_bath_table,1,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,irati,PR,bed_bath_table
3,8283cb8792b84009b158a085333ee8b3,delivered,2018-04-20 08:18:07,2018-04-24 18:14:55,2018-04-23 19:21:04,2018-05-08 16:35:47,2018-05-23,8944a074bce81688fe43eed3b0a4ff1b,4,recomendo,...,2018-05-10 12:13:05,1c47e837f2780378d8485d04ab83a45d,health_beauty,1,094ced053e257ae8cae57205592d6712,ribeirao preto,SP,limoeiro,PE,health_beauty
4,4a5cc9b4e332e03d76bf553a7f2fa5d3,delivered,2017-12-15 23:33:32,2017-12-18 00:33:04,2017-12-19 01:35:09,2018-01-03 23:13:15,2018-01-16,c7e18295d6bec5500da0d9d2ca18ab82,3,None,...,2018-01-07 15:14:14,c2ccc78b5a924096d5274ca6a3118e47,health_beauty,1,709e16e2b25c7474d980076c6bfc4806,birigui,SP,belford roxo,RJ,health_beauty
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98660,c5334d330e36d2a810a7a13c72e135ee,delivered,2018-02-23 08:38:47,2018-02-23 09:20:21,2018-02-27 16:55:50,2018-03-02 19:48:41,2018-03-09,7eeb2cbff212e6c311994d274bc1a149,5,None,...,2018-03-06 12:41:39,1d3f55c00acdb29bd959f2760abc8b7d,auto,1,8581055ce74af1daba164fdbd55a40de,guarulhos,SP,sao paulo,SP,auto
98661,fa4414ea1a5c591f38b0d62663879b17,delivered,2017-11-26 16:50:52,2017-11-27 14:18:30,2017-11-27 22:24:53,2017-11-28 18:33:10,2017-12-11,07a1d72676ab129e2be3b5714ab08a3b,1,None,...,2017-12-05 22:12:33,8f8cb7e4a7f16d339f87f8aa2711a003,toys,1,850f4f8af5ea87287ac68de36e29107f,sao paulo,SP,campinas,SP,toys
98662,c577af5e9d5dbecf0ac65ff28cb8ad45,delivered,2018-08-09 20:25:54,2018-08-10 20:25:11,2018-08-14 15:54:00,2018-08-22 17:09:03,2018-09-05,f5690b65d36177ca3c07f98b2b490004,1,Faltando itens,...,2018-08-27 13:27:14,0615dcf981da53a5ca8777cd6a80361b,home_construction,1,213b25e6f54661939f11710a6fddb871,salto,SP,treze tilias,SC,home_construction
98663,110f23195a2ddb2ccaad30b37bfacbbf,delivered,2017-12-01 16:34:59,2017-12-01 16:49:31,2017-12-08 21:28:46,2017-12-26 19:34:24,2018-01-03,3390d5f386750545dba5deea21fbde3b,5,None,...,2017-12-27 21:52:00,eefb750a45a4ba505ffd4813ecaf2b18,furniture_living_room,1,870d0118f7a9d85960f29ad89d5d989a,pocos de caldas,MG,cariacica,ES,furniture_living_room


In [33]:
df_service_eda.describe()


,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,review_creation_date,review_answer_timestamp,product_count
count,98665,98652,98663,98657,98665,98665.000000,98665,98665,98665.000000
mean,2018-01-02 11:40:02.847706,2018-01-02 22:58:48.161486,2018-01-05 16:48:14.909003,2018-01-14 22:24:39.031310,2018-01-26 06:41:08.352506,4.127644,2018-01-14 17:11:58.613490,2018-01-17 20:45:37.395662,1.099255
min,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-10-04 00:00:00,1.000000,2016-10-06 00:00:00,2016-10-07 18:32:28,1.000000
25%,2017-09-14 13:57:21,2017-09-14 22:25:18.250000,2017-09-18 19:05:10,2017-09-26 16:48:05,2017-10-05 00:00:00,4.000000,2017-09-27 00:00:00,2017-09-29 12:12:07,1.000000
50%,2018-01-21 13:28:36,2018-01-22 14:03:06,2018-01-24 18:52:39,2018-02-02 21:33:30,2018-02-16 00:00:00,5.000000,2018-02-03 00:00:00,2018-02-06 21:05:12,1.000000
75%,2018-05-06 18:49:55,2018-05-07 16:55:38.250000,2018-05-09 10:13:00,2018-05-16 17:24:34,2018-05-28 00:00:00,5.000000,2018-05-17 00:00:00,2018-05-20 20:13:32,1.000000
max,2018-08-29 15:00:37,2018-08-29 15:10:26,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-10-25 00:00:00,5.000000,2018-08-31 00:00:00,2018-10-29 12:27:35,20.000000
std,NaN,NaN,NaN,NaN,NaN,1.308843,NaN,NaN,0.451793


In [34]:
df_service_eda.dtypes


order_id                                   object
order_status                               object
order_purchase_timestamp           datetime64[us]
order_approved_at                  datetime64[us]
order_delivered_carrier_date       datetime64[us]
order_delivered_customer_date      datetime64[us]
order_estimated_delivery_date      datetime64[us]
review_id                                  object
review_score                                int64
review_comment_title                       object
review_comment_message                     object
review_creation_date               datetime64[us]
review_answer_timestamp            datetime64[us]
product_id                                 object
product_category_name_english              object
product_count                               int64
seller_id                                  object
seller_city                                object
seller_state                               object
customer_city                              object


In [35]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = { 
              'order_id': 'category',
              'order_status': 'category',
              'order_approved_at': 'datetime64[s]',
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_delivered_carrier_date': 'datetime64[s]', 
              'order_delivered_customer_date': 'datetime64[s]',
              'order_estimated_delivery_date': 'datetime64[s]',
              'review_id': 'category',
              'review_score': 'Int16',
              'review_comment_title': 'category',
              'review_comment_message': 'category',
              'review_creation_date': 'datetime64[s]',
              'review_answer_timestamp': 'datetime64[s]',
              'product_id': 'category',
              'product_category_name_english': 'category',
              'product_count': 'Int16',
              'seller_id': 'category',
              'seller_city': 'category',
              'seller_state': 'category',
              'customer_city': 'category',
              'customer_state': 'category', 
              
              }
df_service_eda = df_service_eda.astype(col_dtypes)
df_service_eda.dtypes

order_id                                category
order_status                            category
order_purchase_timestamp           datetime64[s]
order_approved_at                  datetime64[s]
order_delivered_carrier_date       datetime64[s]
order_delivered_customer_date      datetime64[s]
order_estimated_delivery_date      datetime64[s]
review_id                               category
review_score                               Int16
review_comment_title                    category
review_comment_message                  category
review_creation_date               datetime64[s]
review_answer_timestamp            datetime64[s]
product_id                              category
product_category_name_english           category
product_count                              Int16
seller_id                               category
seller_city                             category
seller_state                            category
customer_city                           category
customer_state      

In [36]:
df_service_eda.isna().sum()

order_id                               0
order_status                           0
order_purchase_timestamp               0
order_approved_at                     13
order_delivered_carrier_date           2
order_delivered_customer_date          8
order_estimated_delivery_date          0
review_id                              0
review_score                           0
review_comment_title               86959
review_comment_message             58096
review_creation_date                   0
review_answer_timestamp                0
product_id                             0
product_category_name_english          0
product_count                          0
seller_id                              0
seller_city                            0
seller_state                           0
customer_city                          0
customer_state                         0
product_category_name_english_1        0
dtype: int64

In [37]:
df_service_eda['order_approved_at'] = df_service_eda['order_approved_at'].fillna(
    pd.to_datetime(df_service_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_service_eda[
    df_service_eda[['order_delivered_carrier_date', 'order_delivered_customer_date']].isna().any(axis=1)
].head(15)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,...,review_answer_timestamp,product_id,product_category_name_english,product_count,seller_id,seller_city,seller_state,customer_city,customer_state,product_category_name_english_1
4516,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30,bb311d9562ecbefc8e4be756d8999892,5,NaN,...,2018-07-10 11:38:13,e7d5464b94c9a5963f7c686fc80145ad,watches_gifts,1,58f1a6197ed863543e0136bdedb3fce2,conselheiro lafaiete,MG,sumare,SP,watches_gifts
4851,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16,c0dd6bec0375c376f044af102118526f,5,Entrega super rápida.,...,2018-06-29 16:26:37,2167c8f6252667c0eb9edd51520706a1,industry_commerce_and_business,1,0bb738e4d789e63e2267697c42d35a2d,sao roque,SP,quadra,SP,industry_commerce_and_business
50678,2aa91108853cecb43c84a5dc5b277475,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14,e945d1831a3d98008913fc31dcbb804d,5,NaN,...,2017-10-17 10:56:02,44c2baf621113fa7ac95fa06b4afbc68,furniture_decor,1,3f2af2670e104d1bcb54022274daeac5,terra boa,PR,indaiatuba,SP,furniture_decor
51055,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30,25e11638a3d01a87e8e62338a39eee28,5,NaN,...,2018-07-11 19:27:46,e7d5464b94c9a5963f7c686fc80145ad,watches_gifts,1,58f1a6197ed863543e0136bdedb3fce2,conselheiro lafaiete,MG,pindamonhangaba,SP,watches_gifts
60846,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18,f48c6c944a5d52dcca8ac5c4ec417cf2,5,NaN,...,2017-12-19 04:15:39,a50acd33ba7a8da8e9db65094fa990a4,auto,1,8581055ce74af1daba164fdbd55a40de,guarulhos,SP,cerquilho,SP,auto
73857,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26,0d4c56af896dd6eb9de8edbaa1902d22,1,Péssimo,...,2018-06-16 13:55:00,a2a7efc985315e86d4f0f705701b342b,computers_accessories,1,ed4acab38528488b65a9a9c603ff024a,sao paulo,SP,guarulhos,SP,computers_accessories
87976,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23,4e755f114e50d33b9ac6a56e0d7d3ea9,5,NaN,...,2017-06-27 01:49:04,30b5b5635a79548a48d04162d971848f,sports_leisure,1,f9bbdd976532d50b7816d285a22bd01e,sao paulo,SP,porto alegre,RS,sports_leisure
96985,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24,ee2d30652e2f7fc00861074f795f5bf0,5,Excelente!,...,2018-07-07 18:48:09,ec165cd31c50585786ffda6feff5d0a6,toys,1,8bdd8e3fd58bafa48af76b2c5fd71974,sao paulo,SP,sao carlos,SP,toys
97330,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19,d055795a562efffefe47ef81e5435322,5,Muito bom,...,2018-07-06 20:30:17,55bfa0307d7a46bed72c492259921231,books_general_interest,1,343e716476e3748b069f980efbaa294e,campinas,SP,ribeirao pires,SP,books_general_interest


In [38]:
df_service_eda['order_delivered_customer_date'] = df_service_eda['order_delivered_customer_date'].fillna(
    pd.to_datetime(df_service_eda['order_estimated_delivery_date'])
)

df_service_eda['order_delivered_carrier_date'] = df_service_eda['order_delivered_carrier_date'].fillna(
    pd.to_datetime(df_service_eda['order_approved_at']) + pd.Timedelta(days=5)
)
df_service_eda[
    df_service_eda[['order_delivered_carrier_date', 'order_delivered_customer_date']].isna().any(axis=1)
].head(15)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,...,review_answer_timestamp,product_id,product_category_name_english,product_count,seller_id,seller_city,seller_state,customer_city,customer_state,product_category_name_english_1


In [39]:
df_service_eda.describe()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,review_creation_date,review_answer_timestamp,product_count
count,98665,98665,98665,98665,98665,98665.0,98665,98665,98665.0
mean,2018-01-02 11:40:02,2018-01-02 21:57:29,2018-01-05 16:43:40,2018-01-14 22:37:27,2018-01-26 06:41:08,4.127644,2018-01-14 17:11:58,2018-01-17 20:45:37,1.099255
min,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-10-04 00:00:00,1.0,2016-10-06 00:00:00,2016-10-07 18:32:28,1.0
25%,2017-09-14 13:57:21,2017-09-14 21:45:17,2017-09-18 19:02:38,2017-09-26 16:48:13,2017-10-05 00:00:00,4.0,2017-09-27 00:00:00,2017-09-29 12:12:07,1.0
50%,2018-01-21 13:28:36,2018-01-22 14:00:56,2018-01-24 18:48:44,2018-02-02 21:39:55,2018-02-16 00:00:00,5.0,2018-02-03 00:00:00,2018-02-06 21:05:12,1.0
75%,2018-05-06 18:49:55,2018-05-07 16:53:20,2018-05-09 10:13:00,2018-05-16 17:29:13,2018-05-28 00:00:00,5.0,2018-05-17 00:00:00,2018-05-20 20:13:32,1.0
max,2018-08-29 15:00:37,2018-08-29 15:10:26,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-10-25 00:00:00,5.0,2018-08-31 00:00:00,2018-10-29 12:27:35,20.0
std,NaN,NaN,NaN,NaN,NaN,1.308843,NaN,NaN,0.451793


In [40]:
print("Gesamte Duplikate:", df_service_eda.duplicated().sum())


Gesamte Duplikate: 0


In [41]:
order_id = '895ab968e7bb0d5659d16cd74cd1650c'

# 1. Filter auf diese Order_ID
order_data = df_service_eda[(df_service_eda['order_id'] == order_id)]

order_data.head(63).sort_values('order_purchase_timestamp', ascending=True)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,...,review_answer_timestamp,product_id,product_category_name_english,product_count,seller_id,seller_city,seller_state,customer_city,customer_state,product_category_name_english_1
36155,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2017-08-10 11:58:14,2017-08-14 12:46:18,2017-08-30,eef5dbca8d37dfce6db7d7b16dd0525e,5,NaN,...,2017-08-17 22:17:55,ebf9bc6cd600eadd681384e3116fda85,bed_bath_table,2,822166ed1e47908f7cfb49946d03c726,tres rios,RJ,sao paulo,SP,bed_bath_table
75909,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2017-08-10 11:58:14,2017-08-14 12:46:18,2017-08-30,eef5dbca8d37dfce6db7d7b16dd0525e,5,NaN,...,2017-08-17 22:17:55,5ddab10d5e0a23acb99acf56b62b3276,housewares,1,3d0cd21d41671c46f82cd11176bf7277,joinville,SC,sao paulo,SP,housewares


In [42]:
con.execute("CREATE TABLE service_analyse AS SELECT * FROM df_service_eda")


In [43]:
sql("SHOW TABLES")

,name
0,customer_rfm
1,customers
2,geolocation
3,order_items
4,order_payments
5,order_reviews
6,orders
7,product_category
8,product_category_name_translation
9,products


In [44]:
con.close()